# Pipeline ELT — Censo da Educação Superior (INEP) 2022–2024
**Grupo 6 · Projeto de Integração**

**Versão para Windows + VS Code + ambiente virtual `.venv` + PostgreSQL local.**

**Ideia do ELT:** os CSVs são carregados *crus* (tudo como texto) no banco e **todas as transformações são feitas em SQL dentro do banco**.

| Camada | Schema | O que contém |
|---|---|---|
| Bruta | `raw` | Uma tabela por arquivo/ano, colunas = cabeçalho do CSV, tudo `TEXT` |
| Preparação | `stg` | Tabelas tipadas e padronizadas (`stg.cursos`, `stg.ies`), reserva étnica unificada |
| Referência | `ref` | Dicionários código → texto (iguais nos 3 anos, seção 5 do perfilamento) |
| Estrela | `dw` | `fato_oferta_curso` + `dim_*` |

O banco é o mesmo do ETL (`censo_superior`), que grava o seu resultado no schema `etl`. A camada `dw` usa **os mesmos nomes de tabelas, colunas e rótulos** do schema `etl`. Assim, as consultas da pasta `analises/` rodam nos dois caminhos, e a seção 9 compara os dois resultados linha a linha.

## 0. Parâmetros

In [ ]:
from pathlib import Path
import os
from getpass import getpass

ANOS = [2022, 2023, 2024]

# Localiza a raiz do repositório mesmo quando o notebook é aberto dentro de notebooks/ELT.
def localizar_raiz_projeto():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    for pasta in candidatos:
        if (pasta / "data" / "raw").is_dir():
            return pasta
    # Fallback: a pasta atual é considerada a raiz; o teste de arquivos abaixo dará uma mensagem clara.
    return Path.cwd()

ROOT_DIR = localizar_raiz_projeto()
DATA_DIR = ROOT_DIR / "data" / "raw"

def arq_cursos(ano):
    return DATA_DIR / str(ano) / f"MICRODADOS_CADASTRO_CURSOS_{ano}.CSV"

def arq_ies(ano):
    return DATA_DIR / str(ano) / f"MICRODADOS_ED_SUP_IES_{ano}.CSV"

# Conexão PostgreSQL local no Windows.
# Defina PGPASSWORD no Windows se preferir não informar a senha toda vez.
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = os.getenv("PGPORT", "5432")
PGUSER = os.getenv("PGUSER", "postgres")
PGPASSWORD = os.getenv("PGPASSWORD") or getpass("Senha do usuário PostgreSQL: ")
DB_NAME = "censo_superior"

# DSN usado nas próximas células.
DSN = f"host={PGHOST} port={PGPORT} dbname={DB_NAME} user={PGUSER} password={PGPASSWORD}"
DSN_ADMIN = f"host={PGHOST} port={PGPORT} dbname=postgres user={PGUSER} password={PGPASSWORD}"

# Valores de referência (seção 15 do perfilamento).
REF = {
    2022: dict(linhas=573_019, cursos=45_296, ing=4_756_957, mat=9_444_116, ies=2_595),
    2023: dict(linhas=671_610, cursos=46_317, ing=4_994_192, mat=9_977_217, ies=2_580),
    2024: dict(linhas=720_349, cursos=46_150, ing=5_010_613, mat=10_227_266, ies=2_561),
}
REF_SEM_MUN = {2022: 9_603, 2023: 11_039, 2024: 11_778}
REF_DIM4 = {2022: 395, 2023: 459, 2024: 450}

print("Raiz do projeto:", ROOT_DIR)
print("Pasta dos CSVs:", DATA_DIR)


## 1. Ambiente: PostgreSQL local no Windows

Antes de executar:
1. Instale o PostgreSQL para Windows (incluindo o pgAdmin).
2. Confirme que o serviço PostgreSQL está iniciado no Windows (`services.msc`).
3. Tenha a senha do usuário `postgres` definida durante a instalação.
4. Confirme que `ipykernel`, `pandas` e `psycopg2-binary` estão instalados na `.venv`.

Esta etapa cria o banco `censo_superior` se ele ainda não existir (o mesmo usado pelo ETL). Não instala nem reinicia o serviço do PostgreSQL.


In [ ]:
# Dependências Python: execute no kernel da .venv selecionada.
%pip install -q pandas psycopg2-binary ipykernel

# Criar o banco censo se ainda não existir.
import psycopg2
from psycopg2 import sql

admin_conn = psycopg2.connect(DSN_ADMIN)
admin_conn.autocommit = True

with admin_conn.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB_NAME,))
    existe = cur.fetchone() is not None
    if not existe:
        cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(DB_NAME)))
        print(f"Banco '{DB_NAME}' criado.")
    else:
        print(f"Banco '{DB_NAME}' já existe; será reutilizado.")

admin_conn.close()


In [ ]:
import csv
import time
import psycopg2
import pandas as pd

conn = psycopg2.connect(DSN)
conn.autocommit = True

def run(sql, params=None, fetch=False):
    """Executa SQL no banco. Com fetch=True devolve (colunas, linhas)."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        if fetch:
            return [d[0] for d in cur.description], cur.fetchall()

def show(sql, params=None):
    cols, rows = run(sql, params, fetch=True)
    return pd.DataFrame(rows, columns=cols)

run("SELECT 1")
print("Conectado ao PostgreSQL:", PGHOST, PGPORT, DB_NAME)


## 2. E + L — Extração e carga bruta (`raw`)

- Os arquivos são lidos de `data/raw/ANO/` dentro do repositório.
- O cabeçalho é lido de cada CSV (os anos têm 200/202/223 colunas de cursos), então **cada ano ganha a sua tabela** com as colunas que realmente existem.
- Tudo entra como `TEXT`, sem conversão: o tipo é decidido depois, em SQL.
- Encoding `LATIN1` e separador `;`.
- A carga usa `COPY FROM STDIN`, compatível com Windows: o arquivo é enviado pelo cliente Python ao PostgreSQL local.
- O arquivo é aberto em **modo binário** (`"rb"`) e enviado como bytes. Quem converte de `LATIN1` para o UTF-8 do banco é o próprio `COPY`. Abrir o arquivo como texto em latin1 faria o Python converter primeiro e o `COPY` converter de novo, e os acentos seriam gravados corrompidos ("São Paulo" viraria "SÃ£o Paulo") sem que nenhuma contagem acusasse.


In [ ]:
# Conferir todos os arquivos antes de apagar/recriar o schema raw.
arquivos_necessarios = [p for ano in ANOS for p in (arq_cursos(ano), arq_ies(ano))]
faltantes = [p for p in arquivos_necessarios if not p.is_file()]

if faltantes:
    lista = "\n".join(f" - {p}" for p in faltantes)
    raise FileNotFoundError(
        "Não encontrei todos os CSVs esperados. Confira a estrutura data/raw/ANO/:\n" + lista
    )

run("DROP SCHEMA IF EXISTS raw CASCADE; CREATE SCHEMA raw;")

def ler_cabecalho(path):
    with open(path, encoding="latin1", newline="") as f:
        cab = next(csv.reader(f, delimiter=";"))
    cab = [c.replace("\ufeff", "").strip().lower() for c in cab]
    return [c if c else f"col_extra_{i}" for i, c in enumerate(cab)]

def carregar_raw(path, tabela):
    cols = ler_cabecalho(path)
    ddl = ", ".join(f'"{c}" TEXT' for c in cols)
    run(f"DROP TABLE IF EXISTS raw.{tabela}; CREATE TABLE raw.{tabela} ({ddl});")
    t0 = time.time()
    with conn.cursor() as cur, open(path, "rb") as f:  # bytes: quem converte de LATIN1 é o PostgreSQL
        cur.copy_expert(
            f"COPY raw.{tabela} FROM STDIN WITH (FORMAT csv, DELIMITER ';', HEADER true, ENCODING 'LATIN1')",
            f
        )
    n = run(f"SELECT count(*) FROM raw.{tabela}", fetch=True)[1][0][0]
    print(f"raw.{tabela:<12} {len(cols):>3} colunas {n:>9,} linhas ({time.time()-t0:.0f}s)".replace(",", "."))
    return n

for ano in ANOS:
    carregar_raw(arq_cursos(ano), f"cursos_{ano}")
    carregar_raw(arq_ies(ano), f"ies_{ano}")


In [ ]:
# Conferência 1 — o que foi carregado bate com a seção 2 do perfilamento?
for ano in ANOS:
    n_c = run(f"SELECT count(*) FROM raw.cursos_{ano}", fetch=True)[1][0][0]
    n_i = run(f"SELECT count(*) FROM raw.ies_{ano}", fetch=True)[1][0][0]
    assert n_c == REF[ano]["linhas"], f"cursos {ano}: {n_c} != {REF[ano]['linhas']}"
    assert n_i == REF[ano]["ies"],    f"ies {ano}: {n_i} != {REF[ano]['ies']}"
print("OK — contagem de linhas da camada raw confere com o perfilamento.")

## 3. T — Referência (`ref`): dicionários código → texto
Os códigos são idênticos nos 3 anos (seção 5), então um único dicionário serve para todos.

In [ ]:
run('''
DROP SCHEMA IF EXISTS ref CASCADE; CREATE SCHEMA ref;

-- Textos iguais aos do ETL, para que as consultas de análise funcionem nos dois caminhos
CREATE TABLE ref.dic_dimensao (cod SMALLINT PRIMARY KEY, texto VARCHAR(80), modalidade VARCHAR(20));
INSERT INTO ref.dic_dimensao VALUES
 (1,'Presencial no Brasil','Presencial'),
 (2,'EAD no Brasil','EAD'),
 (3,'EAD com dados só em nível Brasil','EAD'),
 (4,'EAD de instituição brasileira no exterior','EAD');

-- Grau não aplicável (cursos de Área Básica de Ingresso e sequenciais) vem de dois jeitos:
-- código 0 em 2022 (fora do dicionário do INEP) e vazio em 2023 e 2024 (convertido para '.' na stg)
CREATE TABLE ref.dic_grau_academico (cod VARCHAR(2) PRIMARY KEY, texto VARCHAR(40));
INSERT INTO ref.dic_grau_academico VALUES
 ('0','Não aplicável'),
 ('1','Bacharelado'),
 ('2','Licenciatura'),
 ('3','Tecnológico'),
 ('4','Bacharelado e Licenciatura'),
 ('.','Não aplicável');

CREATE TABLE ref.dic_nivel_academico (cod VARCHAR(2) PRIMARY KEY, texto VARCHAR(40));
INSERT INTO ref.dic_nivel_academico VALUES
 ('1','Graduação'),('2','Sequencial de Formação Específica');

CREATE TABLE ref.dic_modalidade (cod VARCHAR(2) PRIMARY KEY, texto VARCHAR(20));
INSERT INTO ref.dic_modalidade VALUES ('1','Presencial'),('2','EAD');

CREATE TABLE ref.dic_organizacao (cod VARCHAR(2) PRIMARY KEY, texto VARCHAR(60));
INSERT INTO ref.dic_organizacao VALUES
 ('1','Universidade'),('2','Centro Universitário'),('3','Faculdade'),
 ('4','Instituto Federal'),
 ('5','Centro Federal de Educação Tecnológica');

-- rede é DERIVADA da categoria (seção 12): 1,2,3,7 = Pública; 4,5 = Privada
CREATE TABLE ref.dic_categoria (cod VARCHAR(2) PRIMARY KEY, texto VARCHAR(60), rede VARCHAR(10));
INSERT INTO ref.dic_categoria VALUES
 ('1','Pública Federal','Pública'),
 ('2','Pública Estadual','Pública'),
 ('3','Pública Municipal','Pública'),
 ('7','Especial','Pública'),
 ('4','Privada com fins lucrativos','Privada'),
 ('5','Privada sem fins lucrativos','Privada');

-- TP_REDE (só existe no arquivo de IES a partir de 2023): usado apenas para validar a derivação
CREATE TABLE ref.dic_tp_rede (cod VARCHAR(2) PRIMARY KEY, texto VARCHAR(10));
INSERT INTO ref.dic_tp_rede VALUES ('1','Pública'),('2','Privada');
''')
print("Dicionários criados.")

## 4. T — Preparação (`stg`): tipagem, padronização e reserva étnica
Regras aplicadas (todas em SQL):
- `NULLIF(btrim(x),'')` → vazio vira `NULL`; `QT_*` → `INT`; `IN_*` → `BOOLEAN`.
- **Reserva étnica unificada** (`qt_ing_reserva_etnica`): coluna `QT_ING_RVETNICO` em 2022/2023; soma de `RVPPI + RVQUILO + RVPOVT` em 2024. É uma **aproximação** (seção 8), documentada.
- Código de município `0` (2022) e nulo (2023/24) são mantidos aqui; o tratamento acontece ao montar a `dim_localidade`/fato.
- Grau acadêmico em branco → `'.'` (não aplicável).
- Em 2022, o mesmo caso de grau não aplicável vem com o código `0` (1.407 linhas, quase todas de cursos ABI); o dicionário `ref` traduz `0` e `'.'` como "Não aplicável".
- `CO_CINE_ROTULO` vem com aspas dentro do valor (`"0011A09"`): as aspas são removidas com `btrim`.

In [ ]:
run('''
DROP SCHEMA IF EXISTS stg CASCADE; CREATE SCHEMA stg;

CREATE TABLE stg.cursos (
  nu_ano_censo SMALLINT NOT NULL, co_curso INT NOT NULL, no_curso VARCHAR(255),
  co_ies INT, tp_dimensao SMALLINT NOT NULL,
  co_municipio INT, no_municipio VARCHAR(100),
  co_uf SMALLINT, sg_uf CHAR(2), no_uf VARCHAR(50),
  co_regiao SMALLINT, no_regiao VARCHAR(20), in_capital BOOLEAN,
  co_cine_rotulo VARCHAR(10), no_cine_rotulo VARCHAR(255),
  co_cine_area_geral VARCHAR(10), no_cine_area_geral VARCHAR(255),
  co_cine_area_especifica VARCHAR(10), no_cine_area_especifica VARCHAR(255),
  co_cine_area_detalhada VARCHAR(10), no_cine_area_detalhada VARCHAR(255),
  tp_grau_academico VARCHAR(2), tp_nivel_academico VARCHAR(2), tp_modalidade_ensino VARCHAR(2),
  in_gratuito BOOLEAN,
  qt_vg_total INT, qt_inscrito_total INT, qt_ing INT, qt_mat INT, qt_conc INT,
  qt_ing_fies INT, qt_ing_prounii INT, qt_ing_prounip INT,
  qt_ing_preta INT, qt_ing_parda INT, qt_ing_indigena INT,
  qt_ing_reserva_vaga INT, qt_ing_reserva_etnica INT
);
''')

# Lista de colunas igual para os 3 anos; só a expressão de reserva étnica muda.
def select_cursos(ano, reserva_expr):
    i  = lambda c: f"NULLIF(btrim({c}),'')::int"
    t  = lambda c: f"NULLIF(btrim({c}),'')"
    b  = lambda c: f"(NULLIF(btrim({c}),'')= '1')"
    return f'''
INSERT INTO stg.cursos
SELECT
  {i('nu_ano_censo')}::smallint, {i('co_curso')}, {t('no_curso')},
  {i('co_ies')}, {i('tp_dimensao')}::smallint,
  {i('co_municipio')}, {t('no_municipio')},
  {i('co_uf')}::smallint, {t('sg_uf')}, {t('no_uf')},
  {i('co_regiao')}::smallint, {t('no_regiao')}, {b('in_capital')},
  NULLIF(btrim(co_cine_rotulo, ' "'),''), {t('no_cine_rotulo')},
  {t('co_cine_area_geral')}, {t('no_cine_area_geral')},
  {t('co_cine_area_especifica')}, {t('no_cine_area_especifica')},
  {t('co_cine_area_detalhada')}, {t('no_cine_area_detalhada')},
  COALESCE({t('tp_grau_academico')}, '.'), {t('tp_nivel_academico')}, {t('tp_modalidade_ensino')},
  {b('in_gratuito')},
  {i('qt_vg_total')}, {i('qt_inscrito_total')}, {i('qt_ing')}, {i('qt_mat')}, {i('qt_conc')},
  {i('qt_ing_fies')}, {i('qt_ing_prounii')}, {i('qt_ing_prounip')},
  {i('qt_ing_preta')}, {i('qt_ing_parda')}, {i('qt_ing_indigena')},
  {i('qt_ing_reserva_vaga')}, {reserva_expr}
FROM raw.cursos_{ano};'''

RESERVA = {
  2022: "NULLIF(btrim(qt_ing_rvetnico),'')::int",
  2023: "NULLIF(btrim(qt_ing_rvetnico),'')::int",
  2024: ("COALESCE(NULLIF(btrim(qt_ing_rvppi),'')::int,0)"
         " + COALESCE(NULLIF(btrim(qt_ing_rvquilo),'')::int,0)"
         " + COALESCE(NULLIF(btrim(qt_ing_rvpovt),'')::int,0)"),
}
for ano in ANOS:
    run(select_cursos(ano, RESERVA[ano]))
print(show("SELECT nu_ano_censo, count(*) linhas, count(DISTINCT co_curso) cursos FROM stg.cursos GROUP BY 1 ORDER BY 1"))

In [ ]:
# stg.ies — rede NÃO vem do arquivo (só existe a partir de 2023): é derivada da categoria na dim_ies.
# in_comunitaria / in_confessional só existem de 2023 em diante: em 2022 ficam NULL.
run('''
CREATE TABLE stg.ies (
  nu_ano_censo SMALLINT NOT NULL, co_ies INT NOT NULL, no_ies VARCHAR(255), sg_ies VARCHAR(30),
  tp_organizacao_academica VARCHAR(2), tp_categoria_administrativa VARCHAR(2), tp_rede VARCHAR(2),
  in_comunitaria BOOLEAN, in_confessional BOOLEAN,
  no_municipio_ies VARCHAR(100), sg_uf_ies CHAR(2), no_regiao_ies VARCHAR(20), in_capital_ies BOOLEAN
);''')

def tem_coluna(tabela, coluna):
    return run('''SELECT 1 FROM information_schema.columns
                  WHERE table_schema='raw' AND table_name=%s AND column_name=%s''',
               (tabela, coluna), fetch=True)[1] != []

for ano in ANOS:
    tb = f"ies_{ano}"
    t = lambda c: f"NULLIF(btrim({c}),'')"
    opt  = lambda c: t(c) if tem_coluna(tb, c) else "NULL"
    optb = lambda c: f"({t(c)}='1')" if tem_coluna(tb, c) else "NULL::boolean"
    run(f'''
    INSERT INTO stg.ies
    SELECT {t('nu_ano_censo')}::int::smallint, {t('co_ies')}::int, {t('no_ies')}, {t('sg_ies')},
           {t('tp_organizacao_academica')}, {t('tp_categoria_administrativa')}, {opt('tp_rede')},
           {optb('in_comunitaria')}, {optb('in_confessional')},
           {t('no_municipio_ies')}, {t('sg_uf_ies')}, {t('no_regiao_ies')}, {optb('in_capital_ies')}
    FROM raw.{tb};''')
print(show("SELECT nu_ano_censo, count(*) ies, count(DISTINCT co_ies) distintas FROM stg.ies GROUP BY 1 ORDER BY 1"))

## 5. Conferência dos totais (seção 15) — antes de modelar
Os somatórios de `QT_ING` e `QT_MAT` precisam bater exatamente com os valores de referência. Essa é também a base da conferência ETL × ELT.

In [ ]:
df = show('''
SELECT nu_ano_censo ano, count(*) linhas, count(DISTINCT co_curso) cursos,
       sum(qt_ing)::bigint ing, sum(qt_mat)::bigint mat
FROM stg.cursos GROUP BY 1 ORDER BY 1''')
display(df)
for _, r in df.iterrows():
    ref = REF[int(r.ano)]
    assert (r.linhas, r.cursos, r.ing, r.mat) == (ref["linhas"], ref["cursos"], ref["ing"], ref["mat"]), f"Totais de {int(r.ano)} não batem: {dict(r)}"
print("OK — totais dos 3 anos reproduzem o perfilamento.")

# Reserva étnica 2024 (seção 8): 79.969 + 2.314 + 758 = 83.041
r24 = run("SELECT sum(qt_ing_reserva_etnica) FROM stg.cursos WHERE nu_ano_censo=2024", fetch=True)[1][0][0]
assert r24 == 83_041, r24
print("OK — reserva étnica 2024 =", r24)

## 6. Testes de qualidade na camada `stg` (falham o pipeline se algo mudar)

In [ ]:
def testar(nome, sql, esperado_zero=True):
    n = run(sql, fetch=True)[1][0][0]
    status = "OK " if (n == 0) == esperado_zero else "FALHA"
    print(f"[{status}] {nome}: {n}")
    assert (n == 0) == esperado_zero, nome

# 6.1 chave natural da fato é única nos 3 anos (seção 11)
testar("duplicatas em (ano, curso, município, dimensão)", '''
 SELECT count(*) FROM (SELECT 1 FROM stg.cursos
   GROUP BY nu_ano_censo, co_curso, COALESCE(co_municipio,-999), tp_dimensao HAVING count(*)>1) x''')

# 6.2 todos os códigos têm tradução nos dicionários (código novo = erro explícito)
testar("TP_DIMENSAO sem tradução",  "SELECT count(*) FROM stg.cursos c LEFT JOIN ref.dic_dimensao d ON d.cod=c.tp_dimensao WHERE d.cod IS NULL")
testar("grau acadêmico sem tradução","SELECT count(*) FROM stg.cursos c LEFT JOIN ref.dic_grau_academico d ON d.cod=c.tp_grau_academico WHERE d.cod IS NULL")
testar("nível acadêmico sem tradução","SELECT count(*) FROM stg.cursos c LEFT JOIN ref.dic_nivel_academico d ON d.cod=c.tp_nivel_academico WHERE d.cod IS NULL")
testar("modalidade sem tradução",   "SELECT count(*) FROM stg.cursos c LEFT JOIN ref.dic_modalidade d ON d.cod=c.tp_modalidade_ensino WHERE d.cod IS NULL")
testar("organização sem tradução",  "SELECT count(*) FROM stg.ies i LEFT JOIN ref.dic_organizacao d ON d.cod=i.tp_organizacao_academica WHERE d.cod IS NULL")
testar("categoria administrativa inesperada (rede)", "SELECT count(*) FROM stg.ies i LEFT JOIN ref.dic_categoria d ON d.cod=i.tp_categoria_administrativa WHERE d.cod IS NULL")

# 6.3 rede derivada da categoria == TP_REDE onde existe (2023 e 2024) — seção 12
testar("rede derivada ≠ TP_REDE", '''
 SELECT count(*) FROM stg.ies i
 JOIN ref.dic_categoria c ON c.cod=i.tp_categoria_administrativa
 JOIN ref.dic_tp_rede r   ON r.cod=i.tp_rede
 WHERE c.rede <> r.texto''')

# 6.4 (ano, IES) único e toda IES dos cursos existe no arquivo de IES do mesmo ano
testar("IES duplicada no mesmo ano", "SELECT count(*) FROM (SELECT 1 FROM stg.ies GROUP BY nu_ano_censo, co_ies HAVING count(*)>1) x")
testar("cursos apontando para IES inexistente no ano", '''
 SELECT count(*) FROM stg.cursos c LEFT JOIN stg.ies i ON i.co_ies=c.co_ies AND i.nu_ano_censo=c.nu_ano_censo
 WHERE i.co_ies IS NULL''')

# 6.5 consistência intra-ano dos atributos do curso (seção 14)
testar("curso com >1 valor de atributo dentro do ano", '''
 SELECT count(*) FROM (SELECT 1 FROM stg.cursos GROUP BY nu_ano_censo, co_curso
   HAVING count(DISTINCT co_ies)>1 OR count(DISTINCT co_cine_rotulo)>1 OR count(DISTINCT no_curso)>1
       OR count(DISTINCT tp_grau_academico)>1 OR count(DISTINCT tp_nivel_academico)>1
       OR count(DISTINCT tp_modalidade_ensino)>1 OR count(DISTINCT in_gratuito)>1) x''')

# 6.6 linhas presenciais (dimensão 1) sempre têm município válido; dimensão 4 nunca tem
testar("dimensão 1 sem município", "SELECT count(*) FROM stg.cursos WHERE tp_dimensao=1 AND COALESCE(co_municipio,0)=0")
testar("dimensão 4 com município", "SELECT count(*) FROM stg.cursos WHERE tp_dimensao=4 AND COALESCE(co_municipio,0)<>0")
testar("CINE rótulo nulo", "SELECT count(*) FROM stg.cursos WHERE co_cine_rotulo IS NULL")
testar("CINE rótulo com aspas", "SELECT count(*) FROM stg.cursos WHERE co_cine_rotulo LIKE '%\"%'")

# 6.7 acentos gravados corretamente: as sequências abaixo só aparecem quando o latin1 é convertido duas vezes
testar("acento corrompido em nome de município ou curso", """
 SELECT count(*) FROM stg.cursos
 WHERE no_municipio ~ 'Ã[£§©¡³ªº­µ¢´]' OR no_curso ~ 'Ã[£§©¡³ªº­µ¢´]'""")

## 7. T — Modelo estrela (`dw`)
Estrutura conforme o modelo do grupo (perfilamento). `dim_curso` e `dim_ies` guardam **uma linha por (código, ano)**, de modo que a fato aponta para a versão do próprio ano, sem datas de validade (seção 13). A mantenedora fica fora do modelo.

Os nomes de tabelas e colunas, a ordem das colunas, os rótulos e a ordem das chaves substitutas são **os mesmos do schema `etl`**. Isso permite rodar as mesmas consultas de análise nos dois caminhos e comparar as tabelas linha a linha (seção 9). As colunas ficam em minúsculas, sem prefixos `ds_`/`no_`, como no ETL.

In [ ]:
run('''
DROP SCHEMA IF EXISTS dw CASCADE; CREATE SCHEMA dw;

CREATE TABLE dw.dim_tempo (
  sk_tempo INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  ano SMALLINT NOT NULL UNIQUE);

CREATE TABLE dw.dim_dimensao (
  sk_dimensao INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  co_dimensao SMALLINT NOT NULL UNIQUE,
  dimensao_geografica VARCHAR(80) NOT NULL,
  modalidade VARCHAR(20) NOT NULL);

CREATE TABLE dw.dim_curso (
  sk_curso INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  co_curso INT NOT NULL, ano SMALLINT NOT NULL,
  nome_curso VARCHAR(255), grau_academico VARCHAR(40), modalidade VARCHAR(20),
  nivel_academico VARCHAR(40), gratuito VARCHAR(5),
  UNIQUE (co_curso, ano));

CREATE TABLE dw.dim_ies (
  sk_ies INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  co_ies INT NOT NULL, ano SMALLINT NOT NULL,
  nome_ies VARCHAR(255), sigla_ies VARCHAR(30),
  organizacao_academica VARCHAR(60), categoria_administrativa VARCHAR(60), rede VARCHAR(10),
  comunitaria VARCHAR(15), confessional VARCHAR(15),
  municipio_sede VARCHAR(100), uf_sede CHAR(2), regiao_sede VARCHAR(20),
  UNIQUE (co_ies, ano));

CREATE TABLE dw.dim_area_cine (
  sk_area_cine INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  co_cine_rotulo VARCHAR(10) NOT NULL UNIQUE, rotulo_cine VARCHAR(255),
  area_geral VARCHAR(255), area_especifica VARCHAR(255), area_detalhada VARCHAR(255));

-- códigos negativos = membros especiais (seção 10 do perfilamento), iguais aos do ETL
CREATE TABLE dw.dim_localidade (
  sk_localidade INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  co_municipio INT NOT NULL UNIQUE, municipio VARCHAR(100),
  sg_uf CHAR(2), uf VARCHAR(50), regiao VARCHAR(20), capital VARCHAR(15));
''')

# Em todas as dimensões, as linhas entram ordenadas pela chave natural, como no ETL,
# para que as chaves substitutas sejam as mesmas nos dois caminhos.

# --- dim_tempo e dim_dimensao
run("INSERT INTO dw.dim_tempo (ano) SELECT DISTINCT nu_ano_censo FROM stg.cursos ORDER BY 1")
run('''INSERT INTO dw.dim_dimensao (co_dimensao, dimensao_geografica, modalidade)
       SELECT cod, texto, modalidade FROM ref.dic_dimensao ORDER BY cod''')

# --- dim_curso: 1 linha por (curso, ano) — atributos são estáveis dentro do ano (seção 14)
run('''
INSERT INTO dw.dim_curso (co_curso, ano, nome_curso, grau_academico, modalidade, nivel_academico, gratuito)
SELECT DISTINCT ON (c.co_curso, c.nu_ano_censo)
   c.co_curso, c.nu_ano_censo, c.no_curso, g.texto, m.texto, n.texto,
   CASE WHEN c.in_gratuito THEN 'Sim' ELSE 'Não' END
FROM stg.cursos c
JOIN ref.dic_grau_academico g ON g.cod = c.tp_grau_academico
JOIN ref.dic_nivel_academico n ON n.cod = c.tp_nivel_academico
JOIN ref.dic_modalidade m ON m.cod = c.tp_modalidade_ensino
ORDER BY c.co_curso, c.nu_ano_censo''')

# --- dim_ies: 1 linha por (IES, ano); rede derivada da categoria (seção 12)
#     comunitária e confessional não existem em 2022: "Não informado"
run('''
INSERT INTO dw.dim_ies (co_ies, ano, nome_ies, sigla_ies, organizacao_academica,
   categoria_administrativa, rede, comunitaria, confessional, municipio_sede, uf_sede, regiao_sede)
SELECT i.co_ies, i.nu_ano_censo, i.no_ies, i.sg_ies, o.texto, c.texto, c.rede,
       CASE WHEN i.in_comunitaria IS NULL THEN 'Não informado' WHEN i.in_comunitaria THEN 'Sim' ELSE 'Não' END,
       CASE WHEN i.in_confessional IS NULL THEN 'Não informado' WHEN i.in_confessional THEN 'Sim' ELSE 'Não' END,
       i.no_municipio_ies, i.sg_uf_ies, i.no_regiao_ies
FROM stg.ies i
JOIN ref.dic_organizacao o ON o.cod = i.tp_organizacao_academica
JOIN ref.dic_categoria  c ON c.cod = i.tp_categoria_administrativa
ORDER BY i.co_ies, i.nu_ano_censo''')

# --- dim_area_cine: 1 linha por rótulo, com o nome mais recente (seção 14)
run('''
INSERT INTO dw.dim_area_cine (co_cine_rotulo, rotulo_cine, area_geral, area_especifica, area_detalhada)
SELECT co_cine_rotulo, no_cine_rotulo, no_cine_area_geral, no_cine_area_especifica, no_cine_area_detalhada
FROM (
  SELECT DISTINCT ON (co_cine_rotulo)
     co_cine_rotulo, no_cine_rotulo, no_cine_area_geral, no_cine_area_especifica, no_cine_area_detalhada
  FROM stg.cursos
  WHERE co_cine_rotulo IS NOT NULL
  ORDER BY co_cine_rotulo, nu_ano_censo DESC, co_curso) x
ORDER BY co_cine_rotulo COLLATE "C"''')

# --- dim_localidade: 3 membros especiais (seção 10) + municípios reais (valor mais recente)
run('''
INSERT INTO dw.dim_localidade (co_municipio, municipio, sg_uf, uf, regiao, capital)
SELECT * FROM (
  SELECT -3, 'Não informado', 'NI', 'Não informado', 'Não informado', 'Não se aplica'
  UNION ALL SELECT -2, 'Exterior', 'EX', 'Exterior', 'Exterior', 'Não se aplica'
  UNION ALL SELECT -1, 'Sem município (EAD nível Brasil)', 'BR', 'Não se aplica', 'Não se aplica', 'Não se aplica'
  UNION ALL
  SELECT * FROM (
    SELECT DISTINCT ON (co_municipio)
       co_municipio, no_municipio, sg_uf, no_uf, no_regiao,
       CASE WHEN in_capital THEN 'Sim' ELSE 'Não' END
    FROM stg.cursos
    WHERE co_municipio IS NOT NULL AND co_municipio <> 0
    ORDER BY co_municipio, nu_ano_censo DESC) m
) x (co_municipio, municipio, sg_uf, uf, regiao, capital)
ORDER BY co_municipio''')

for t in ["dim_tempo", "dim_dimensao", "dim_curso", "dim_ies", "dim_area_cine", "dim_localidade"]:
    print(f"dw.{t:<15}", run(f"SELECT count(*) FROM dw.{t}", fetch=True)[1][0][0])

### Fato `fato_oferta_curso`
**Granularidade:** uma linha por *curso × município × dimensão geográfica × ano* (seção 11).

Regra de localidade, definida por `TP_DIMENSAO` e não pelo código (seção 10), igual à do ETL:

| Código na `dim_localidade` | Membro | Regra |
|---|---|---|
| -2 | Exterior | dimensão 4 |
| -1 | Sem município (EAD nível Brasil) | dimensão 3, município nulo ou 0 |
| -3 | Não informado | demais linhas sem município (poucas linhas da dimensão 2) |
| código IBGE | o próprio município | todas as outras |

Medidas nulas viram 0 (aditivas). As colunas seguem a mesma ordem da fato do ETL. A chave primária e as chaves estrangeiras são criadas logo depois da carga, para validar todas as linhas de uma vez.

In [ ]:
run('''
-- As chaves (primária e estrangeiras) são criadas DEPOIS da carga: assim o banco valida as quase
-- 2 milhões de linhas de uma vez, em vez de checar cada linha durante o INSERT, o que é bem mais rápido.
CREATE TABLE dw.fato_oferta_curso (
  sk_tempo INT NOT NULL, sk_dimensao INT NOT NULL, sk_curso INT NOT NULL,
  sk_ies INT NOT NULL, sk_area_cine INT NOT NULL, sk_localidade INT NOT NULL,
  qt_vg_total INT, qt_inscrito_total INT, qt_ing INT, qt_mat INT, qt_conc INT,
  qt_ing_fies INT, qt_ing_prounii INT, qt_ing_prounip INT,
  qt_ing_preta INT, qt_ing_parda INT, qt_ing_indigena INT,
  qt_ing_reserva_vaga INT, qt_ing_reserva_etnica INT
);

INSERT INTO dw.fato_oferta_curso
SELECT dt.sk_tempo, dd.sk_dimensao, dc.sk_curso, di.sk_ies, da.sk_area_cine, dl.sk_localidade,
       COALESCE(s.qt_vg_total,0), COALESCE(s.qt_inscrito_total,0), COALESCE(s.qt_ing,0),
       COALESCE(s.qt_mat,0), COALESCE(s.qt_conc,0), COALESCE(s.qt_ing_fies,0),
       COALESCE(s.qt_ing_prounii,0), COALESCE(s.qt_ing_prounip,0), COALESCE(s.qt_ing_preta,0),
       COALESCE(s.qt_ing_parda,0), COALESCE(s.qt_ing_indigena,0),
       COALESCE(s.qt_ing_reserva_vaga,0), COALESCE(s.qt_ing_reserva_etnica,0)
FROM (
  SELECT c.*,
         CASE WHEN c.tp_dimensao = 4 THEN -2
              WHEN COALESCE(c.co_municipio,0) = 0 AND c.tp_dimensao = 3 THEN -1
              WHEN COALESCE(c.co_municipio,0) = 0 THEN -3
              ELSE c.co_municipio END AS co_municipio_dw
  FROM stg.cursos c) s
JOIN dw.dim_tempo      dt ON dt.ano = s.nu_ano_censo
JOIN dw.dim_dimensao   dd ON dd.co_dimensao = s.tp_dimensao
JOIN dw.dim_curso      dc ON dc.co_curso = s.co_curso AND dc.ano = s.nu_ano_censo
JOIN dw.dim_ies        di ON di.co_ies   = s.co_ies   AND di.ano = s.nu_ano_censo
JOIN dw.dim_area_cine  da ON da.co_cine_rotulo = s.co_cine_rotulo
JOIN dw.dim_localidade dl ON dl.co_municipio = s.co_municipio_dw;

ALTER TABLE dw.fato_oferta_curso
  ADD PRIMARY KEY (sk_tempo, sk_curso, sk_localidade, sk_dimensao),   -- = (ano, curso, município, dimensão)
  ADD FOREIGN KEY (sk_tempo)      REFERENCES dw.dim_tempo,
  ADD FOREIGN KEY (sk_dimensao)   REFERENCES dw.dim_dimensao,
  ADD FOREIGN KEY (sk_curso)      REFERENCES dw.dim_curso,
  ADD FOREIGN KEY (sk_ies)        REFERENCES dw.dim_ies,
  ADD FOREIGN KEY (sk_area_cine)  REFERENCES dw.dim_area_cine,
  ADD FOREIGN KEY (sk_localidade) REFERENCES dw.dim_localidade;

CREATE INDEX ix_fato_localidade ON dw.fato_oferta_curso (sk_localidade);
CREATE INDEX ix_fato_ies        ON dw.fato_oferta_curso (sk_ies);
CREATE INDEX ix_fato_area       ON dw.fato_oferta_curso (sk_area_cine);
CREATE INDEX ix_fato_dimensao   ON dw.fato_oferta_curso (sk_dimensao);
ANALYZE;
''')
print("fato:", run("SELECT count(*) FROM dw.fato_oferta_curso", fetch=True)[1][0][0], "linhas")

## 8. Validação final da camada `dw`
Nenhuma linha pode ser perdida entre `raw → stg → dw`, e os totais precisam bater com a referência.

In [ ]:
df = show('''
SELECT t.ano, count(*) linhas, count(DISTINCT c.co_curso) cursos,
       sum(f.qt_ing)::bigint ing, sum(f.qt_mat)::bigint mat
FROM dw.fato_oferta_curso f
JOIN dw.dim_tempo t USING (sk_tempo) JOIN dw.dim_curso c USING (sk_curso)
GROUP BY 1 ORDER BY 1''')
display(df)
for _, r in df.iterrows():
    ref = REF[int(r.ano)]
    assert (r.linhas, r.cursos, r.ing, r.mat) == (ref["linhas"], ref["cursos"], ref["ing"], ref["mat"]), f"Fato {int(r.ano)} não bate: {dict(r)}"
print("OK — linhas, cursos distintos, ingressantes e matrículas idênticos ao perfilamento (nenhuma linha perdida).")

# Localidade: os três membros especiais somam exatamente as linhas sem município do perfilamento (seção 10)
loc = show('''
SELECT t.ano,
  count(*) FILTER (WHERE l.co_municipio = -2) exterior,
  count(*) FILTER (WHERE l.co_municipio = -1) sem_municipio,
  count(*) FILTER (WHERE l.co_municipio = -3) nao_informado
FROM dw.fato_oferta_curso f
JOIN dw.dim_tempo t USING (sk_tempo) JOIN dw.dim_localidade l USING (sk_localidade)
GROUP BY 1 ORDER BY 1''')
display(loc)
for _, r in loc.iterrows():
    a = int(r.ano)
    assert r.exterior == REF_DIM4[a], (a, r.exterior)
    assert r.exterior + r.sem_municipio + r.nao_informado == REF_SEM_MUN[a], (a, r.exterior + r.sem_municipio + r.nao_informado)
print("OK — membros especiais da dim_localidade conferem.")

# Acentos: um município conhecido precisa aparecer com o nome correto
assert run("SELECT count(*) FROM dw.dim_localidade WHERE municipio = 'São Paulo'", fetch=True)[1][0][0] == 1
print("OK — acentos gravados corretamente (ex.: São Paulo).")

# Conferência 2023 por dimensão (seção 9)
display(show('''
SELECT d.co_dimensao, sum(qt_ing)::bigint qt_ing, sum(qt_mat)::bigint qt_mat,
       sum(qt_vg_total)::bigint qt_vg_total, sum(qt_inscrito_total)::bigint qt_inscrito_total
FROM dw.fato_oferta_curso f JOIN dw.dim_dimensao d USING (sk_dimensao) JOIN dw.dim_tempo t USING (sk_tempo)
WHERE t.ano = 2023 GROUP BY 1 ORDER BY 1'''))

## 9. Conferência ETL × ELT
O notebook `01_ETL.ipynb` grava o resultado do caminho ETL no schema `etl` do **mesmo** banco. Como a camada `dw` tem os mesmos nomes, colunas, rótulos e chaves substitutas, as tabelas dos dois caminhos podem ser comparadas **linha a linha**:

- `linhas ETL` e `linhas ELT`: quantidade de linhas de cada tabela;
- `só no ETL` e `só no ELT`: linhas que existem em um schema e não no outro (`EXCEPT ALL`, que também considera linhas repetidas).

Se o ETL ainda não foi executado neste banco, a conferência é pulada.

In [ ]:
ETL_SCHEMA = "etl"   # schema gravado pelo 01_ETL.ipynb no mesmo banco
TABELAS = ["dim_tempo", "dim_dimensao", "dim_curso", "dim_ies", "dim_area_cine", "dim_localidade", "fato_oferta_curso"]

etl_existe = run("SELECT 1 FROM information_schema.schemata WHERE schema_name = %s", (ETL_SCHEMA,), fetch=True)[1] != []
if etl_existe:
    conta = lambda sql: run(sql, fetch=True)[1][0][0]
    linhas = []
    for t in TABELAS:
        so_etl = conta(f"SELECT count(*) FROM (SELECT * FROM {ETL_SCHEMA}.{t} EXCEPT ALL SELECT * FROM dw.{t}) x")
        so_elt = conta(f"SELECT count(*) FROM (SELECT * FROM dw.{t} EXCEPT ALL SELECT * FROM {ETL_SCHEMA}.{t}) x")
        linhas.append((t, conta(f"SELECT count(*) FROM {ETL_SCHEMA}.{t}"), conta(f"SELECT count(*) FROM dw.{t}"),
                       so_etl, so_elt, "IDÊNTICA" if so_etl == so_elt == 0 else "DIVERGE"))
    comparacao = pd.DataFrame(linhas, columns=["tabela", "linhas ETL", "linhas ELT", "só no ETL", "só no ELT", "resultado"])
    display(comparacao)
    assert (comparacao["resultado"] == "IDÊNTICA").all(), "ETL e ELT divergem"
    print("OK — as sete tabelas são idênticas linha a linha nos dois caminhos.")
else:
    print(f"Conferência ETL × ELT pulada: o schema {ETL_SCHEMA} não existe neste banco (rode o 01_ETL.ipynb antes).")

## 10. Consultas de exemplo sobre o modelo
Lembrete (seção 11): cursos sempre com `COUNT(DISTINCT co_curso)`, nunca por número de linhas; vagas de EAD só existem em nível Brasil (não quebrar por região).

In [ ]:
# Ingressantes e cursos distintos por região e ano (apenas presencial/EAD por polo — recortes com localidade real)
display(show('''
SELECT t.ano, l.regiao,
       count(DISTINCT c.co_curso) cursos, sum(f.qt_ing) ingressantes
FROM dw.fato_oferta_curso f
JOIN dw.dim_tempo t USING (sk_tempo) JOIN dw.dim_localidade l USING (sk_localidade)
JOIN dw.dim_curso c USING (sk_curso)
WHERE l.co_municipio > 0
GROUP BY 1,2 ORDER BY 1,2'''))

In [ ]:
# Ingressantes por rede e ano, com % de reserva étnica (aproximação — ver seção 8 do perfilamento)
display(show('''
SELECT t.ano, i.rede, sum(f.qt_ing) ingressantes,
       round(100.0*sum(f.qt_ing_reserva_etnica)/NULLIF(sum(f.qt_ing),0),2) pct_reserva_etnica
FROM dw.fato_oferta_curso f
JOIN dw.dim_tempo t USING (sk_tempo) JOIN dw.dim_ies i USING (sk_ies)
GROUP BY 1,2 ORDER BY 1,2'''))